In [1]:
import pandas as pd
import json
import os
from rdkit import Chem
import rootutils

root_dir = rootutils.setup_root(".",
                                indicator=".project-root",
                                pythonpath=True)

from benchmarks.max_frag import maxfrag_accuracy, maxfrag

In [2]:
df_gt = pd.read_csv(root_dir / "data" / "uspto_50k_test_250.csv")

df_gt.drop(columns=['reaction_type', "cluster_id"], inplace=True)
df_gt.rename(columns={
    "input": "products",
    "output": "reactants"
},
             inplace=True)


In [3]:
def canonicalize_smiles(smiles):
    return Chem.MolToSmiles(Chem.MolFromSmiles(smiles))


df_gt['products'] = df_gt['products'].apply(canonicalize_smiles)
df_gt['reactants'] = df_gt['reactants'].apply(canonicalize_smiles)


In [4]:
# iterate over the df_gt

results = {}
for index, row in df_gt.iterrows():
    products = row['products']
    if products not in results:
        results[products] = {
            "claude37sonnet:Pistachio_100+": [],
            "claude3opus:Pistachio_100+": [],
            "claude45sonnet:Pistachio_100+": [],
            "claude4opus:Pistachio_100+": [],
            "claude4sonnet:Pistachio_100+": [],
        }


In [5]:
# load results from json files from folders of each model

master_folder = root_dir / "results" / "single_step"
count = {}
prod_max_len = 0

for model_name in os.listdir(master_folder):
    model_folder = master_folder / model_name / "run_1"
    print(model_name)
    count[model_name] = 0
    for file in os.listdir(model_folder):
        if file.endswith(".json"):
            count[model_name] += 1
            with open(model_folder / file, "r") as f:
                data = json.load(f)
                products = canonicalize_smiles(data['molecule'])
                reactants = data['result']['children'][0]['children'][0]
                try:
                    results[products][model_name] = reactants
                except:
                    print(products)


claude37sonnet:Pistachio_100+
claude3opus:Pistachio_100+
claude45sonnet:Pistachio_100+


claude4opus:Pistachio_100+
claude4sonnet:Pistachio_100+


In [6]:
# add results to df_gt
df_gt['results'] = df_gt['products'].apply(lambda x: results[x])
# split the models in resuls into their own columns
df_gt = pd.concat([df_gt, df_gt['results'].apply(pd.Series)], axis=1)


In [7]:
def calc_maxfrag(gt_reactants, pred_reactants):
    gt_reactants_mols = gt_reactants
    res = False
    for pred in pred_reactants:
        pred_str = ".".join(pred)
        if maxfrag_accuracy(gt_reactants_mols, pred_str):
            res = True
            break
    return res


def calc_all_correct(gt_reactants, pred_reactants, products):
    res = False
    gt_reactants_mols = gt_reactants.split(".")
    if len(pred_reactants) == 0:
        return False
    # print(pred_reactants, gt_reactants_mols)
    for pred in pred_reactants:
        # check if all preds are in gt_reactants
        pred_canon = [
            Chem.MolToSmiles(Chem.MolFromSmiles(preds)) for preds in pred
        ]
        # print(pred_canon, gt_reactants_mols, products)
        if (gt_reactants_mols in pred_canon) or (pred_canon
                                                 in gt_reactants_mols):

            res = True
            break
    return res


def calc_any_correct_with_products(gt_reactants, pred_reactants, products):
    res = False
    gt_reactants_mols = gt_reactants.split(".")
    if len(pred_reactants) == 0:
        return False
    for pred in pred_reactants:
        pred_canon = [
            Chem.MolToSmiles(Chem.MolFromSmiles(preds)) for preds in pred
        ]
        if any(react in pred_canon for react in gt_reactants_mols):
            res = True
            break
    return res


In [ ]:
# claude37sonnet
df_gt['maxfrag_claude37sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_maxfrag(row['reactants'], row[
        'claude37sonnet:Pistachio_100+']),
    axis=1)
df_gt['all_correct_claude37sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_all_correct(row['reactants'], row[
        'claude37sonnet:Pistachio_100+'], row['products']),
    axis=1)
df_gt['any_correct_claude37sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_any_correct_with_products(
        row['reactants'], row['claude37sonnet:Pistachio_100+'], row['products']
    ),
    axis=1)

# claude4sonnet
df_gt['maxfrag_claude4sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_maxfrag(row['reactants'], row[
        'claude4sonnet:Pistachio_100+']),
    axis=1)
df_gt['all_correct_claude4sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_all_correct(row['reactants'], row[
        'claude4sonnet:Pistachio_100+'], row['products']),
    axis=1)
df_gt['any_correct_claude4sonnet:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_any_correct_with_products(
        row['reactants'], row['claude4sonnet:Pistachio_100+'], row['products']
    ),
    axis=1)

# claude3opus
df_gt['maxfrag_claude3opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_maxfrag(row['reactants'], row['claude3opus:Pistachio_100+'
                                                   ]),
    axis=1)
df_gt['all_correct_claude3opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_all_correct(row['reactants'], row[
        'claude3opus:Pistachio_100+'], row['products']),
    axis=1)
df_gt['any_correct_claude3opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_any_correct_with_products(
        row['reactants'], row['claude3opus:Pistachio_100+'], row['products']),
    axis=1)

# claude4opus
df_gt['maxfrag_claude4opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_maxfrag(row['reactants'], row['claude4opus:Pistachio_100+'
                                                   ]),
    axis=1)
df_gt['all_correct_claude4opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_all_correct(row['reactants'], row[
        'claude4opus:Pistachio_100+'], row['products']),
    axis=1)
df_gt['any_correct_claude4opus:Pistachio_100+'] = df_gt.apply(
    lambda row: calc_any_correct_with_products(
        row['reactants'], row['claude4opus:Pistachio_100+'], row['products']),
    axis=1)


[15:02:42] WARNING: not removing hydrogen atom without neighbors
[15:02:42] WARNING: not removing hydrogen atom without neighbors
[15:02:42] WARNING: not removing hydrogen atom without neighbors
[15:02:42] WARNING: not removing hydrogen atom without neighbors


In [13]:
df_gt.sum()

/tmp/ipykernel_1933460/2646491751.py:1: FutureWarning: The default value of numeric_only in DataFrame.sum is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  df_gt.sum()


products                                     CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(...
reactants                                    CC(C)(C)[Si](C)(C)OCCOc1ccc(C=O)nc1Br.CS(=O)c1...
claude37sonnet:Pistachio_100+                [[Brc1ccc(OCCO[Si](C)(C)C(C)(C)C)nc1C=O, CS(=O...
claude3opus:Pistachio_100+                   [[CS(=O)c1cccc(Br)c1.OCCOc1ccc(C#N)cn1], [CS(=...
claude45sonnet:Pistachio_100+                                                               []
claude4opus:Pistachio_100+                   [[CSc1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(C)...
claude4sonnet:Pistachio_100+                 [[CS(=O)c1cccc(Br)c1, B(O)(O)c1nc(CO)ccc1OCCO[...
maxfrag_claude37sonnet:Pistachio_100+                                                       75
all_correct_claude37sonnet:Pistachio_100+                                                    0
any_correct_claude37sonnet:Pistachio_100+                                                  107
maxfrag_claude3opus:Pistachio_100+                

In [10]:
df_gt['products'][0]

'CS(=O)c1cccc(-c2nc(C=O)ccc2OCCO[Si](C)(C)C(C)(C)C)c1'

In [11]:
# reactants_mols = [Chem.MolFromSmiles(react) for react in reactants.split(".")]
# for product in products:
#     products_mols = [Chem.MolFromSmiles(react) for react in product.split(".")]
#     products_maxfrag = max(products_mols, key=lambda x: x.GetNumAtoms())
#     print(Chem.MolToSmiles(products_maxfrag))

# reactants_maxfrag = max(reactants_mols, key=lambda x: x.GetNumAtoms())
# products_maxfrag = max(products_mols, key=lambda x: x.GetNumAtoms())